# 05 — Entrenamiento TabDDPM

**Fase 2 — Modelo generativo tabular con difusión gaussiana**  
Referencia: Kotelnikov et al., *TabDDPM: Modelling Tabular Data with Diffusion Models* (2022).

A diferencia de CTGAN/TVAE (Notebook 04), TabDDPM no usa adversarial training:
aprende a revertir un proceso de difusión gaussiana que corrompe los datos progresivamente.
Esto lo hace más estable en entrenamiento y más capaz de capturar distribuciones multimodales,
aunque requiere más tiempo de inferencia (T pasos de denoising).

Pasos:
1. Preprocesamiento: codificar features mixtas en un espacio continuo
2. Instanciar el denoiser (MLP con time embeddings) y el cosine scheduler
3. Entrenar con Adam + cosine LR decay
4. Generar muestras condicionales por mortalidad
5. Post-procesar (inverse transform) y guardar

## 0. Imports y configuración

In [1]:
import sys, warnings, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

ROOT      = Path("..")
PROCESSED = ROOT / "data" / "processed"
SYNTHETIC = ROOT / "data" / "synthetic"
MODELS    = ROOT / "models"
REPORTS   = ROOT / "reports"
for d in [SYNTHETIC, MODELS, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

# Añadir src/ al path para importar el módulo del modelo
sys.path.insert(0, str(ROOT / "src"))
from models.tabddpm import TabDDPMDenoiser, CosineScheduler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")

Device: cuda
  GPU: NVIDIA RTX A4500


## 1. Carga y clasificación de columnas

In [ ]:
tab = pd.read_parquet(PROCESSED / "tabular_48h.parquet")
print(f"Shape: {tab.shape}")

TARGET  = "hospital_expire_flag"
id_cols = [c for c in tab.columns if c.endswith("_id")]

# El target se extrae como label de conditioning; no entra en el proceso de difusión
feature_cols = [c for c in tab.columns if c not in id_cols + [TARGET]]

# dtype != object evita clasificar columnas string ('M'/'F') como binarias numéricas
binary_cols = [
    c for c in feature_cols
    if tab[c].dropna().nunique() <= 2 and tab[c].dtype != object
]
cat_cols = [
    c for c in feature_cols
    if c not in binary_cols and tab[c].dtype == object
]
num_cols = [
    c for c in feature_cols
    if c not in binary_cols + cat_cols
]

print(f"\nFeatures totales:      {len(feature_cols)}")
print(f"  Numéricas:           {len(num_cols)}")
print(f"  Binarias:            {len(binary_cols)}")
print(f"  Categóricas texto:   {len(cat_cols)}  → {cat_cols}")
print(f"\nTarget (label):        {TARGET}  ({tab[TARGET].mean()*100:.1f}% positivos)")

## 2. Preprocesamiento para difusión

El proceso de difusión gaussiana opera en un espacio continuo.
Todas las features se codifican en un vector continuo de dimensión fija:

- **Numéricas**: StandardScaler → z-scores (media=0, std=1)
- **Binarias 0/1**: → {−1, +1} (simétricas respecto al origen, coherente con ruido gaussiano)
- **Categóricas de texto**: OrdinalEncoder → normalizar a [−1, +1]

El target se extrae por separado y se pasa como clase de conditioning al modelo.

In [3]:
class TabularPreprocessor:
    """
    Codifica un DataFrame de features mixtas en un array numpy continuo
    apto para difusión gaussiana. Soporta fit/transform/inverse_transform.
    """

    def __init__(self, num_cols: list, binary_cols: list, cat_cols: list):
        self.num_cols    = num_cols
        self.binary_cols = binary_cols
        self.cat_cols    = cat_cols
        self.input_dim   = len(num_cols) + len(binary_cols) + len(cat_cols)
        self.num_scaler  = StandardScaler() if num_cols else None
        self.cat_encoder = (
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            if cat_cols else None
        )

    def fit(self, df: pd.DataFrame) -> "TabularPreprocessor":
        if self.num_scaler:
            self.num_scaler.fit(df[self.num_cols])
        if self.cat_encoder:
            self.cat_encoder.fit(df[self.cat_cols])
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        parts = []
        if self.num_scaler:
            parts.append(self.num_scaler.transform(df[self.num_cols]).astype(np.float32))
        if self.binary_cols:
            parts.append(df[self.binary_cols].values.astype(np.float32) * 2 - 1)
        if self.cat_encoder:
            enc = self.cat_encoder.transform(df[self.cat_cols]).astype(np.float32)
            for i in range(enc.shape[1]):
                n = len(self.cat_encoder.categories_[i])
                enc[:, i] = enc[:, i] / max(n - 1, 1) * 2 - 1
            parts.append(enc)
        return np.concatenate(parts, axis=1)

    def fit_transform(self, df: pd.DataFrame) -> np.ndarray:
        return self.fit(df).transform(df)

    def inverse_transform(self, arr: np.ndarray) -> pd.DataFrame:
        result = {}
        idx = 0

        if self.num_scaler:
            n   = len(self.num_cols)
            inv = self.num_scaler.inverse_transform(arr[:, idx:idx + n])
            for i, col in enumerate(self.num_cols):
                result[col] = inv[:, i]
            idx += n

        if self.binary_cols:
            n   = len(self.binary_cols)
            raw = arr[:, idx:idx + n]
            for i, col in enumerate(self.binary_cols):
                result[col] = np.clip(np.round((raw[:, i] + 1) / 2), 0, 1).astype(int)
            idx += n

        if self.cat_encoder:
            n   = len(self.cat_cols)
            enc = arr[:, idx:idx + n].copy()
            for i in range(n):
                n_cats   = len(self.cat_encoder.categories_[i])
                enc[:, i] = np.clip(np.round((enc[:, i] + 1) / 2 * (n_cats - 1)), 0, n_cats - 1)
            inv = self.cat_encoder.inverse_transform(enc.astype(int))
            for i, col in enumerate(self.cat_cols):
                result[col] = inv[:, i]

        return pd.DataFrame(result)


# Ajustar y transformar
preprocessor = TabularPreprocessor(num_cols, binary_cols, cat_cols)
X = preprocessor.fit_transform(tab)
y = tab[TARGET].values.astype(np.int64)

print(f"Tensor de entrada: {X.shape}  (estancias × features codificadas)")
print(f"Rango aproximado:  [{X.min():.2f}, {X.max():.2f}]  (esperado ~[-3, 3] tras z-score)")
print(f"Labels:            {np.bincount(y)}  (0=superviviente, 1=fallecido)")

ValueError: could not convert string to float: 'M'

## 3. Modelo y scheduler

### Justificación de hiperparámetros

| Parámetro | Valor | Justificación |
|---|---|---|
| `T` | 1000 | Pasos de difusión; estándar en DDPM (Ho et al. 2020) |
| `hidden_dims` | (512,512,512,512) | 4 capas × 512 neuronas; mayor capacidad que CTGAN/TVAE para compensar la ausencia de adversarial training |
| `time_emb_dim` | 128 | Mismo orden que `embedding_dim` de CTGAN/TVAE para comparabilidad |
| `dropout` | 0.0 | El proceso de difusión actúa como regularizador implícito; dropout adicional penaliza la reconstrucción |
| `batch_size` | 4096 | Minibatches grandes estabilizan la loss de difusión (más ruidosa que GAN/VAE) |
| `lr` | 3e-4 | Estándar para Adam en modelos de difusión |
| `N_EPOCHS` | 1000 | Suficiente para convergencia; se monitoriza la loss cada 100 épocas |
| cosine schedule `s` | 0.008 | Parámetro de suavizado de Nichol & Dhariwal (2021); reduce ruido excesivo al final de la cadena |

In [ ]:
T         = 1000
N_EPOCHS  = 1000
BATCH     = 4096
LR        = 3e-4

INPUT_DIM = preprocessor.input_dim
print(f"input_dim: {INPUT_DIM}")

model = TabDDPMDenoiser(
    input_dim=INPUT_DIM,
    hidden_dims=(512, 512, 512, 512),
    time_emb_dim=128,
    num_classes=2,
    dropout=0.0,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {n_params:,}")

diffusion = CosineScheduler(T=T, s=0.008).to(DEVICE)

# Dataset y DataLoader
X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).long()
dataset  = TensorDataset(X_tensor, y_tensor)
loader   = DataLoader(dataset, batch_size=BATCH, shuffle=True, drop_last=True, num_workers=0)

## 4. Entrenamiento

In [ ]:
optimizer  = torch.optim.Adam(model.parameters(), lr=LR)
lr_sched   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

losses = []
t0 = time.time()

for epoch in tqdm(range(1, N_EPOCHS + 1), desc="Entrenando TabDDPM"):
    model.train()
    epoch_loss = 0.0
    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = diffusion.training_loss(model, x_batch, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    lr_sched.step()

    if epoch % 100 == 0:
        elapsed = (time.time() - t0) / 60
        tqdm.write(f"Epoch {epoch:>4}/{N_EPOCHS}  loss: {avg_loss:.5f}  ({elapsed:.1f} min)")

total_time = (time.time() - t0) / 60
print(f"\nEntrenamiento completado en {total_time:.1f} min.")

# Guardar checkpoint
torch.save(
    {"model_state": model.state_dict(), "losses": losses,
     "input_dim": INPUT_DIM, "T": T},
    MODELS / "tabddpm_checkpoint.pt"
)
print("Checkpoint guardado: models/tabddpm_checkpoint.pt")

In [ ]:
# Curva de convergencia
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color="steelblue", linewidth=1.0, label="Train loss (MSE)")

# Media móvil para visualizar la tendencia
window = 20
if len(losses) >= window:
    rolling = pd.Series(losses).rolling(window).mean()
    ax.plot(rolling, color="tomato", linewidth=1.8, label=f"Media móvil ({window} épocas)")

ax.set_xlabel("Época")
ax.set_ylabel("MSE Loss")
ax.set_title("Curva de convergencia — TabDDPM")
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS / "tabddpm_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/tabddpm_loss_curve.png")

## 5. Generación de muestras sintéticas

Se genera condicionado en la distribución real de mortalidad:
n_supervivientes muestras con y=0 y n_fallecidos con y=1.

Nota: la inferencia recorre los T=1000 pasos en sentido inverso;
es más lenta que CTGAN/TVAE pero solo ocurre una vez.

In [ ]:
n_fallecidos     = int(tab[TARGET].sum())
n_supervivientes = len(tab) - n_fallecidos
N_SAMPLES        = len(tab)

y_gen = torch.cat([
    torch.zeros(n_supervivientes, dtype=torch.long),
    torch.ones(n_fallecidos, dtype=torch.long),
]).to(DEVICE)

print(f"Generando {N_SAMPLES:,} muestras (T={T} pasos de denoising)...")
t0 = time.time()

x_gen = diffusion.sample(model, N_SAMPLES, INPUT_DIM, DEVICE, y=y_gen)
x_gen_np = x_gen.cpu().numpy()

print(f"Inferencia completada en {(time.time()-t0)/60:.1f} min.")

# Inverse transform → DataFrame con las features originales
synth_df = preprocessor.inverse_transform(x_gen_np)
synth_df[TARGET] = y_gen.cpu().numpy()

synth_df.to_parquet(SYNTHETIC / "tabddpm_samples.parquet", index=False)
print(f"Guardado: data/synthetic/tabddpm_samples.parquet  {synth_df.shape}")
print(f"Mortalidad generada: {synth_df[TARGET].mean()*100:.1f}%  (real: {tab[TARGET].mean()*100:.1f}%)")

## 6. Validación rápida de distribuciones

In [ ]:
# Cargar también CTGAN/TVAE para comparación conjunta (si existen)
models_available = {"TabDDPM": synth_df}
for name, fname in [("CTGAN", "ctgan_samples.parquet"), ("TVAE", "tvae_samples.parquet")]:
    p = SYNTHETIC / fname
    if p.exists():
        models_available[name] = pd.read_parquet(p)

COLORS = {"TabDDPM": "steelblue", "CTGAN": "tomato", "TVAE": "seagreen"}
VITALS_PLOT = [
    "heart_rate_mean", "sbp_mean", "spo2_mean", "gcs_total_mean",
    "resp_rate_mean", "lactate_mean", "creatinine_mean", "age"
]
vitals_plot = [c for c in VITALS_PLOT if c in tab.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, vitals_plot):
    ax.hist(tab[col].dropna(), bins=60, alpha=0.5, density=True,
            color="gray", label="Real")
    for mname, df_s in models_available.items():
        if col in df_s.columns:
            ax.hist(df_s[col].dropna(), bins=60, alpha=0.45, density=True,
                    color=COLORS.get(mname, "black"), label=mname)
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7)
for j in range(len(vitals_plot), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribuciones marginales — Real vs modelos generativos", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / "tabddpm_marginals.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/tabddpm_marginals.png")

In [ ]:
# Estadísticos descriptivos comparados
key_cols = [c for c in ["heart_rate_mean", "sbp_mean", "lactate_mean",
                         "creatinine_mean", "age", "los"] if c in tab.columns]
rows = []
for col in key_cols:
    row = {"variable": col,
           "real_mean": tab[col].mean(), "real_std": tab[col].std()}
    for mname, df_s in models_available.items():
        if col in df_s.columns:
            row[f"{mname.lower()}_mean"] = df_s[col].mean()
            row[f"{mname.lower()}_std"]  = df_s[col].std()
    rows.append(row)

stats_df = pd.DataFrame(rows).set_index("variable").round(3)
print(stats_df.to_string())
stats_df.to_csv(REPORTS / "tabddpm_stats_comparison.csv")
print("\nGuardado: reports/tabddpm_stats_comparison.csv")

## 7. Resumen

In [ ]:
print("=" * 60)
print("  RESUMEN — Notebook 05")
print("=" * 60)
checks = [
    ("input_dim (features codificadas)", str(INPUT_DIM)),
    ("Parámetros del modelo",            f"{n_params:,}"),
    ("T (pasos de difusión)",            str(T)),
    ("Épocas entrenadas",                str(N_EPOCHS)),
    ("Loss final",                       f"{losses[-1]:.5f}"),
    ("Tiempo entrenamiento",             f"{total_time:.1f} min"),
    ("Muestras generadas",               f"{synth_df.shape}"),
    ("Mortalidad generada",              f"{synth_df[TARGET].mean()*100:.1f}%"),
]
for name, val in checks:
    print(f"  {name:<35} {val}")
print("=" * 60)
print("Listos para notebook 06 (TimeGAN).")